**《Maximum Likelihood Training of Score-Based Diffusion Models》** 

论文原文：[arXiv:2101.09258](https://arxiv.org/pdf/2101.09258)。  
[arXiv](https://arxiv.org/html/2101.09258?utm_source=chatgpt.com)

# 一、论文的核心问题与符号

## 1. 为什么 Song SDE 之后还需要这篇论文？

上一篇 Song SDE（2021）告诉我们：

$$
dX_t=f(X_t,t)dt+g(t)dW_t
$$

可以构造反向 SDE：

$$
dX_t=
[f(X_t,t)-g(t)^2\nabla\log p_t(X_t)]dt
+g(t)d\bar W_t
$$

以及 Probability Flow ODE：

$$
\frac{dX_t}{dt}
=
f(X_t,t)-\frac12g(t)^2\nabla\log p_t(X_t)
$$

同时可以通过 DSM 训练 Score 网络：

$$
s_\theta(x,t)\approx\nabla\log p_t(x)
$$

但问题在于：

**DSM 训练的是 Score 误差，而最大似然训练优化的是概率密度。二者有什么严格联系？**

假设真实数据分布是 $p_0$，模型生成分布是 $p_\theta$。

最大似然等价于：

$$
\min_\theta D_{\mathrm{KL}}(p_0\|p_\theta)
$$

Score Matching 则优化：

$$
\min_\theta
\int_0^T
\lambda(t)
\mathbb E_{p_t}
\left[
\|s_\theta-\nabla\log p_t\|^2
\right]dt
$$

两者显然不是同一个公式。

ScoreFlow 要解决的问题是：

> 能不能证明某种特殊加权的 Score Matching 损失，是生成模型负对数似然的上界？

答案是可以，关键权重为：

$$
\boxed{\lambda(t)=g(t)^2}
$$

## 2. 统一符号

| 符号 | 含义 |
|---|---|
| $x\in\mathbb R^D$ | 一个 $D$ 维数据样本 |
| $p_0(x)$ | 真实数据概率密度，论文也写作 $p(x)$ |
| $p_t(x)$ | 真实数据经前向 SDE 扰动后的边缘密度 |
| $p_{0t}(x_t\mid x_0)$ | 前向 SDE 的条件转移密度 |
| $\pi(x)$ | 生成过程使用的终端先验 |
| $f(x,t)$ | 前向 SDE 漂移 |
| $g(t)$ | 前向 SDE 扩散系数 |
| $s_\theta(x,t)$ | 网络预测的 Score |
| $p_\theta^{\mathrm{SDE}}$ | 近似反向 SDE 在 $t=0$ 产生的分布 |
| $p_\theta^{\mathrm{ODE}}$ | 近似 Probability Flow ODE 在 $t=0$ 产生的分布 |
| $q_0,q_t$ | Theorem 2 中引入的另一组初始分布及其前向边缘 |
| $\mathcal H(p)$ | 微分熵，$-\int p(x)\log p(x)dx$ |

特别注意：

$$
\boxed{
p_\theta^{\mathrm{SDE}}
\quad\text{与}\quad
p_\theta^{\mathrm{ODE}}
}
$$

是两个不同定义的生成模型。

当网络 Score 精确满足相应一致性条件时，它们可以相同；但一般情况下，不能默认：

$$
p_\theta^{\mathrm{SDE}}=p_\theta^{\mathrm{ODE}}
$$

---

# 二、论文第 2–3 节：训练目标与生成模型

## 3. 连续时间 Score Matching

论文公式 (3)：

$$
\boxed{
\mathcal J_{\mathrm{SM}}(\theta;\lambda)
=
\frac12\int_0^T
\lambda(t)
\mathbb E_{X_t\sim p_t}
\left[
\|\nabla\log p_t(X_t)-s_\theta(X_t,t)\|^2
\right]dt
}
\tag{3}
$$

这里：

$$
\lambda(t)>0
$$

是时间权重。

为什么前面有：

$$
\frac12
$$

只是为了后续推导方便，与 MSE 中常用的 $\frac12$ 相同。

该损失的最优解是：

$$
s_\theta^*(x,t)=\nabla\log p_t(x)
$$

但真实边缘 Score：

$$
\nabla\log p_t(x)
$$

通常无法直接计算。

因此需要将它转化成 DSM。

## 4. 从 SM 推导 DSM

设：

$$
X_0\sim p_0
$$

$$
X_t\sim p_{0t}(\cdot\mid X_0)
$$

定义条件 Score：

$$
u_t(X_t,X_0)
=
\nabla_{X_t}\log p_{0t}(X_t\mid X_0)
$$

根据我们之前推导过的 DSM 恒等式：

$$
\boxed{
\mathbb E[u_t(X_t,X_0)\mid X_t=x]
=
\nabla\log p_t(x)
}
$$

现在比较：

$$
\mathbb E\|s_\theta-u_t\|^2
$$

与：

$$
\mathbb E\|s_\theta-\nabla\log p_t\|^2
$$

展开第一项：

$$
\begin{aligned}
\mathbb E\|s_\theta-u_t\|^2
={}&
\mathbb E\|s_\theta\|^2
-2\mathbb E[s_\theta^\top u_t]
+\mathbb E\|u_t\|^2
\end{aligned}
$$

利用条件期望：

$$
\begin{aligned}
\mathbb E[s_\theta^\top u_t]
&=
\mathbb E\left[
s_\theta(X_t,t)^\top
\mathbb E[u_t\mid X_t]
\right]\\
&=
\mathbb E[
s_\theta(X_t,t)^\top\nabla\log p_t(X_t)
]
\end{aligned}
$$

所以：

$$
\begin{aligned}
\mathbb E\|s_\theta-u_t\|^2
={}&
\mathbb E\|s_\theta-\nabla\log p_t\|^2\\
&+
\mathbb E\|u_t\|^2
-\mathbb E\|\nabla\log p_t\|^2
\end{aligned}
$$

后两项与 $\theta$ 无关。

因此：

$$
\boxed{
\mathcal J_{\mathrm{DSM}}(\theta;\lambda)
=
\mathcal J_{\mathrm{SM}}(\theta;\lambda)+C
}
$$

其中：

$$
C
=
\frac12\int_0^T\lambda(t)
\left[
\mathbb E\|u_t\|^2-
\mathbb E\|\nabla\log p_t\|^2
\right]dt
$$

于是论文公式 (4)：

$$
\boxed{
\mathcal J_{\mathrm{DSM}}(\theta;\lambda)
=
\frac12\int_0^T
\lambda(t)
\mathbb E_{p_0(x_0)p_{0t}(x_t\mid x_0)}
\left[
\|s_\theta(x_t,t)-u_t(x_t,x_0)\|^2
\right]dt
}
\tag{4}
$$

可以直接用于训练。

**重要区别：**

$$
\mathcal J_{\mathrm{DSM}}
$$

与：

$$
\mathcal J_{\mathrm{SM}}
$$

不是数值完全相同，而是相差一个与模型参数无关的常数。

因此它们具有相同的优化目标，但不能随意将 DSM 的数值直接当成 KL 上界。[arXiv](https://arxiv.org/html/2101.09258?utm_source=chatgpt.com)

## 5. 两个生成模型的定义

论文第 3 节明确区分了两个模型。

### 5.1 反向 SDE 模型

$$
\boxed{
dX_t=
[f(X_t,t)-g(t)^2s_\theta(X_t,t)]dt
+g(t)d\bar W_t
}
$$

其中：

$$
X_T\sim\pi
$$

积分方向：

$$
T\rightarrow0
$$

最终：

$$
X_0\sim p_\theta^{\mathrm{SDE}}
$$

### 5.2 Probability Flow ODE 模型

$$
\boxed{
\frac{dX_t}{dt}
=
f(X_t,t)-\frac12g(t)^2s_\theta(X_t,t)
}
$$

同样：

$$
X_T\sim\pi
$$

最终：

$$
X_0\sim p_\theta^{\mathrm{ODE}}
$$

对于 ODE，可以通过连续正规化流的变量变换公式计算：

$$
\boxed{
\log p_\theta^{\mathrm{ODE}}(x_0)
=
\log\pi(x_T)
+
\int_0^T\nabla\cdot v_\theta(x_t,t)dt
}
$$

其中：

$$
v_\theta=f-\frac12g^2s_\theta
$$

但直接最大化这一似然，需要在每次训练时执行 ODE 积分及相关梯度计算，计算成本较高。

所以论文选择研究：

**能否使用便宜的 DSM 训练目标，间接优化生成模型的似然？**

---

# 三、Theorem 1：为什么 Likelihood Weighting 是 $g(t)^2$？

这是论文最核心的定理。

## 6. 定理内容

论文公式 (8)：

$$
\boxed{
D_{\mathrm{KL}}
\left(
p_0\|p_\theta^{\mathrm{SDE}}
\right)
\leq
\mathcal J_{\mathrm{SM}}(\theta;g^2)
+
D_{\mathrm{KL}}(p_T\|\pi)
}
\tag{8}
$$

其中：

$$
\boxed{
\mathcal J_{\mathrm{SM}}(\theta;g^2)
=
\frac12\int_0^T
g(t)^2
\mathbb E_{p_t}
\left[
\|\nabla\log p_t-s_\theta\|^2
\right]dt
}
$$

这意味着：

**使用 $g(t)^2$ 加权的 Score Matching 损失，可以控制真实数据分布与反向 SDE 生成分布之间的 KL 散度。**

下面完整推导。

## 7. 第一步：为什么要引入路径分布？

我们希望控制：

$$
D_{\mathrm{KL}}(p_0\|p_\theta^{\mathrm{SDE}})
$$

但 $p_\theta^{\mathrm{SDE}}$ 的概率密度通常不能直接计算。

因此转而研究完整随机过程的概率分布。

定义：

$$
\mathbb P
$$

为真实前向扩散过程的路径测度。

定义：

$$
\mathbb Q_\theta
$$

为模型反向 SDE 的路径测度。

二者都可以视为定义在连续轨迹空间：

$$
\{x_t:0\leq t\leq T\}
$$

上的概率分布。

它们在 $t=0$ 的边缘分别为：

$$
p_0
$$

和：

$$
p_\theta^{\mathrm{SDE}}
$$

现在考虑一个映射：

$$
F(\{x_t\}_{t=0}^T)=x_0
$$

也就是从完整轨迹中只提取最终生成状态。

根据 KL 散度的数据处理不等式：

$$
\boxed{
D_{\mathrm{KL}}(F_\#\mathbb P\|F_\#\mathbb Q_\theta)
\leq
D_{\mathrm{KL}}(\mathbb P\|\mathbb Q_\theta)
}
$$

其中 $F_\#$ 表示通过映射 $F$ 得到的分布。

因此：

$$
\boxed{
D_{\mathrm{KL}}(p_0\|p_\theta^{\mathrm{SDE}})
\leq
D_{\mathrm{KL}}(\mathbb P\|\mathbb Q_\theta)
}
\tag{A}
$$

这一步非常关键。

**终点 KL 不容易计算，但路径 KL 可以通过 SDE 漂移之间的差异计算。**

## 8. 第二步：将路径 KL 拆分

两种反向过程在时间 $T$ 的初始分布分别是：

真实过程：

$$
X_T\sim p_T
$$

模型过程：

$$
\hat X_T\sim\pi
$$

根据 KL 的链式法则：

$$
\boxed{
\begin{aligned}
D_{\mathrm{KL}}(\mathbb P\|\mathbb Q_\theta)
={}&
D_{\mathrm{KL}}(p_T\|\pi)\\
&+
\mathbb E_{X_T\sim p_T}
D_{\mathrm{KL}}
\left(
\mathbb P(\cdot\mid X_T)
\|
\mathbb Q_\theta(\cdot\mid X_T)
\right)
\end{aligned}
}
\tag{B}
$$

这与 DDPM 的 ELBO 分解很相似。

第一项表示：

$$
p_T
$$

与：

$$
\pi
$$

之间的差异。

第二项表示：给定相同的反向初始状态，真实和模型的随机动力学有多大差异。

## 9. 第三步：Girsanov 定理

现在考虑两条反向 SDE。

真实反向 SDE：

$$
dX_t=
[f(X_t,t)-g(t)^2s^*(X_t,t)]dt
+g(t)d\bar W_t
$$

其中：

$$
s^*(x,t)=\nabla\log p_t(x)
$$

模型反向 SDE：

$$
d\hat X_t=
[f(\hat X_t,t)-g(t)^2s_\theta(\hat X_t,t)]dt
+g(t)d\bar W_t
$$

两者的扩散系数相同：

$$
g(t)
$$

但漂移不同。

为避免反向时间符号混淆，令：

$$
\tau=T-t
$$

并定义：

$$
Y_\tau=X_{T-\tau}
$$

真实反向过程在正向时间 $\tau$ 下的漂移为：

$$
b^*(x,\tau)
=
-f(x,T-\tau)+g(T-\tau)^2s^*(x,T-\tau)
$$

模型漂移为：

$$
b_\theta(x,\tau)
=
-f(x,T-\tau)+g(T-\tau)^2s_\theta(x,T-\tau)
$$

因此：

$$
\boxed{
b^*-b_\theta=g(t)^2(s^*-s_\theta)
}
$$

定义归一化漂移差：

$$
\boxed{
u(x,t)
=
\frac{b^*-b_\theta}{g(t)}
=
g(t)[s^*(x,t)-s_\theta(x,t)]
}
$$

现在使用 Girsanov 定理。

对于扩散系数相同、漂移不同的两条 SDE，在适当的绝对连续性和 Novikov 条件下，路径 KL 为：

$$
\boxed{
D_{\mathrm{KL}}(\mathbb P\|\mathbb Q_\theta)
=
D_{\mathrm{KL}}(p_T\|\pi)
+
\frac12
\mathbb E_{\mathbb P}
\int_0^T
\|u(X_t,t)\|^2dt
}
\tag{C}
$$

### 9.1 为什么 Girsanov 会得到平方误差？

这个结果可以从布朗运动的概率测度变换理解。

设：

$$
dY_\tau=b^*(Y_\tau,\tau)d\tau+g(\tau)dW_\tau
$$

另一条 SDE 为：

$$
dY_\tau=b_\theta(Y_\tau,\tau)d\tau+g(\tau)d\widetilde W_\tau
$$

两条 SDE 之间的漂移差可以通过改变布朗运动的测度来表示。

令：

$$
u=\frac{b^*-b_\theta}{g}
$$

在 Girsanov 条件成立时，两个路径测度之间的对数密度比包含：

$$
\int_0^T u^\top dW_\tau
$$

和：

$$
\frac12\int_0^T\|u\|^2d\tau
$$

两部分。

当我们计算 KL：

$$
D_{\mathrm{KL}}(\mathbb P\|\mathbb Q_\theta)
=
\mathbb E_{\mathbb P}
\left[
\log\frac{d\mathbb P}{d\mathbb Q_\theta}
\right]
$$

时，Itô 随机积分的期望为零：

$$
\mathbb E_{\mathbb P}
\left[
\int_0^T u^\top dW_\tau
\right]=0
$$

于是只剩：

$$
\boxed{
\frac12\mathbb E_{\mathbb P}
\int_0^T\|u\|^2d\tau
}
$$

这就是为什么路径 KL 最终变成了漂移误差的平方积分。

### 9.2 代入 Score 误差

因为：

$$
u=g(t)(s^*-s_\theta)
$$

所以：

$$
\|u\|^2
=
g(t)^2\|s^*-s_\theta\|^2
$$

于是：

$$
\boxed{
\begin{aligned}
D_{\mathrm{KL}}(\mathbb P\|\mathbb Q_\theta)
={}&
D_{\mathrm{KL}}(p_T\|\pi)\\
&+
\frac12\int_0^T
g(t)^2
\mathbb E_{p_t}
\left[
\|s^*(X_t,t)-s_\theta(X_t,t)\|^2
\right]dt
\end{aligned}
}
\tag{D}
$$

注意这里的期望是在真实过程的边缘分布 $p_t$ 下计算。

这是因为 KL 的第一个参数是真实路径测度 $\mathbb P$。

## 10. 第四步：得到 Theorem 1

由数据处理不等式：

$$
D_{\mathrm{KL}}(p_0\|p_\theta^{\mathrm{SDE}})
\leq
D_{\mathrm{KL}}(\mathbb P\|\mathbb Q_\theta)
$$

再代入路径 KL：

$$
\boxed{
\begin{aligned}
D_{\mathrm{KL}}(p_0\|p_\theta^{\mathrm{SDE}})
\leq{}&
D_{\mathrm{KL}}(p_T\|\pi)\\
&+
\frac12\int_0^Tg(t)^2
\mathbb E_{p_t}
\left[
\|\nabla\log p_t-s_\theta\|^2
\right]dt
\end{aligned}
}
$$

因此：

$$
\boxed{
\lambda(t)=g(t)^2
}
$$

推导完成。

这就是论文 Theorem 1 的完整逻辑：

$$
\boxed{
\text{终点 KL}
\leq
\text{路径 KL}
=
\text{先验 KL}
+
\text{加权 Score 误差}
}
$$

该证明对应论文附录 A 的公式 (14)–(18)。[arXiv](https://arxiv.org/html/2101.09258?utm_source=chatgpt.com)

---

# 四、Theorem 2：什么时候 KL 上界能够取等号？

Theorem 1 只给出了：

$$
D_{\mathrm{KL}}(p_0\|p_\theta^{\mathrm{SDE}})
\leq
J_{\mathrm{SM}}+D_{\mathrm{KL}}(p_T\|\pi)
$$

接下来作者提出两个问题：

1. 什么时候这个上界能够取等号？
2. 什么时候反向 SDE 和 Probability Flow ODE 的生成分布相同？

Theorem 2 同时回答这两个问题。

## 11. Theorem 2 的条件

假设存在另一个初始分布：

$$
q_0(x)
$$

它与真实数据分布：

$$
p_0(x)
$$

不一定相同。

让二者经过**同一个前向 SDE**：

$$
dX_t=f(X_t,t)dt+g(t)dW_t
$$

得到两组边缘分布：

$$
p_t(x),\qquad q_t(x)
$$

假设满足：

$$
\boxed{q_T=\pi}
$$

并且网络恰好预测：

$$
\boxed{
s_\theta(x,t)=\nabla\log q_t(x)
}
$$

那么论文证明：

$$
\boxed{
p_\theta^{\mathrm{SDE}}
=
p_\theta^{\mathrm{ODE}}
=
q_0
}
\tag{9}
$$

而且：

$$
\boxed{
D_{\mathrm{KL}}(p_0\|q_0)
=
D_{\mathrm{KL}}(p_T\|q_T)
+
\frac12\int_0^Tg(t)^2
\mathbb E_{p_t}
\left[
\|\nabla\log p_t-\nabla\log q_t\|^2
\right]dt
}
\tag{10}
$$

这实际上是一个关于**相对 Fisher 信息**的恒等式。

下面从 Fokker–Planck 方程推导。

## 12. 相对 Fisher 信息是什么？

普通 Fisher 信息的一种形式为：

$$
I(p)=
\mathbb E_p
\left[
\|\nabla\log p(x)\|^2
\right]
$$

而相对 Fisher 信息定义为：

$$
\boxed{
I(p\|q)
=
\mathbb E_p
\left[
\left\|
\nabla\log\frac{p(x)}{q(x)}
\right\|^2
\right]
}
$$

由于：

$$
\nabla\log\frac pq
=
\nabla\log p-\nabla\log q
$$

因此：

$$
\boxed{
I(p\|q)
=
\mathbb E_p
\left[
\|\nabla\log p-\nabla\log q\|^2
\right]
}
$$

这恰好就是 Score Matching 的误差形式。

因此，Theorem 2 的关键是证明：

$$
\boxed{
\frac{d}{dt}D_{\mathrm{KL}}(p_t\|q_t)
=
-\frac12g(t)^2I(p_t\|q_t)
}
$$

也就是说：

**当两个分布经过同一个扩散过程时，它们之间的 KL 散度随时间减小，其下降速度由相对 Fisher 信息决定。**

## 13. 从 Fokker–Planck 推导 KL 的时间导数

定义：

$$
K(t)=D_{\mathrm{KL}}(p_t\|q_t)
$$

即：

$$
K(t)=\int p_t(x)\log\frac{p_t(x)}{q_t(x)}dx
$$

对时间求导：

$$
\begin{aligned}
K'(t)
={}&
\int\partial_tp_t\log\frac{p_t}{q_t}dx\\
&+
\int\partial_tp_t\,dx\\
&-
\int\frac{p_t}{q_t}\partial_tq_t\,dx
\end{aligned}
$$

由于概率密度归一化：

$$
\int p_tdx=1
$$

所以：

$$
\int\partial_tp_tdx=0
$$

于是：

$$
\boxed{
K'(t)
=
\int\partial_tp_t\log\frac{p_t}{q_t}dx
-
\int\frac{p_t}{q_t}\partial_tq_tdx
}
$$

定义：

$$
r_t(x)=\frac{p_t(x)}{q_t(x)}
$$

以及：

$$
a_t(x)=\nabla\log r_t(x)
$$

那么：

$$
a_t=\nabla\log p_t-\nabla\log q_t
$$

### 13.1 分别写出两个 Fokker–Planck 方程

因为两组分布使用同一个前向 SDE：

$$
\partial_tp_t
=
-\nabla\cdot(fp_t)+\frac12g^2\Delta p_t
$$

$$
\partial_tq_t
=
-\nabla\cdot(fq_t)+\frac12g^2\Delta q_t
$$

为了简化，定义概率流：

$$
J_p=fp_t-\frac12g^2\nabla p_t
$$

$$
J_q=fq_t-\frac12g^2\nabla q_t
$$

于是：

$$
\partial_tp_t=-\nabla\cdot J_p
$$

$$
\partial_tq_t=-\nabla\cdot J_q
$$

代入 KL 导数：

$$
K'(t)
=
-\int\nabla\cdot J_p\log r_tdx
+
\int r_t\nabla\cdot J_qdx
$$

对两项分别分部积分，假设边界项消失：

$$
-\int\nabla\cdot J_p\log r_tdx
=
\int J_p^\top\nabla\log r_tdx
$$

$$
\int r_t\nabla\cdot J_qdx
=
-\int J_q^\top\nabla r_tdx
$$

因为：

$$
\nabla r_t=r_t\nabla\log r_t=r_ta_t
$$

所以：

$$
\boxed{
K'(t)
=
\int
(J_p-r_tJ_q)^\top a_tdx
}
$$

### 13.2 计算两个概率流之差

代入：

$$
J_p=fp_t-\frac12g^2\nabla p_t
$$

$$
r_tJ_q=r_tfq_t-\frac12g^2r_t\nabla q_t
$$

因为：

$$
r_tq_t=p_t
$$

所以漂移项抵消：

$$
fp_t-r_tfq_t=0
$$

剩下：

$$
J_p-r_tJ_q
=
-\frac12g^2
\left[
\nabla p_t-r_t\nabla q_t
\right]
$$

利用：

$$
\nabla p_t=p_t\nabla\log p_t
$$

以及：

$$
r_t\nabla q_t
=
p_t\nabla\log q_t
$$

得到：

$$
\begin{aligned}
J_p-r_tJ_q
&=
-\frac12g^2p_t
\left[
\nabla\log p_t-\nabla\log q_t
\right]\\
&=
-\frac12g^2p_ta_t
\end{aligned}
$$

代回：

$$
K'(t)
=
-\frac12g^2
\int p_t\|a_t\|^2dx
$$

因此：

$$
\boxed{
\frac{d}{dt}D_{\mathrm{KL}}(p_t\|q_t)
=
-\frac12g(t)^2
\mathbb E_{p_t}
\left[
\|\nabla\log p_t-\nabla\log q_t\|^2
\right]
}
$$

推导完成。

这是整篇论文最重要的数学恒等式之一。

### 13.3 对时间积分

从：

$$
0
$$

积分到：

$$
T
$$

得到：

$$
\begin{aligned}
D_{\mathrm{KL}}(p_T\|q_T)
-
D_{\mathrm{KL}}(p_0\|q_0)
={}&
-\frac12\int_0^Tg(t)^2
\mathbb E_{p_t}
\left[
\|\nabla\log p_t-\nabla\log q_t\|^2
\right]dt
\end{aligned}
$$

整理：

$$
\boxed{
\begin{aligned}
D_{\mathrm{KL}}(p_0\|q_0)
={}&
D_{\mathrm{KL}}(p_T\|q_T)\\
&+
\frac12\int_0^Tg(t)^2
\mathbb E_{p_t}
\left[
\|\nabla\log p_t-\nabla\log q_t\|^2
\right]dt
\end{aligned}
}
$$

根据 Theorem 2 的条件：

$$
q_T=\pi
$$

$$
s_\theta=\nabla\log q_t
$$

因此：

$$
\boxed{
D_{\mathrm{KL}}(p_0\|q_0)
=
D_{\mathrm{KL}}(p_T\|\pi)
+
\mathcal J_{\mathrm{SM}}(\theta;g^2)
}
$$

这就是论文公式 (10)。

## 14. 为什么此时 SDE 和 ODE 的生成分布相同？

因为网络输出：

$$
s_\theta=\nabla\log q_t
$$

恰好是 $q_t$ 的真实 Score。

于是模型反向 SDE：

$$
dX_t=
[f-g^2s_\theta]dt+gd\bar W_t
$$

就变成：

$$
dX_t=
[f-g^2\nabla\log q_t]dt+gd\bar W_t
$$

这正是以 $q_0$ 为初始分布的前向 SDE 的精确时间反转。

而模型在 $T$ 时刻初始化为：

$$
X_T\sim\pi=q_T
$$

所以反向生成的终点必然满足：

$$
X_0\sim q_0
$$

因此：

$$
\boxed{
p_\theta^{\mathrm{SDE}}=q_0
}
$$

另一方面，Probability Flow ODE：

$$
\frac{dX_t}{dt}
=
f-\frac12g^2\nabla\log q_t
$$

与同一个前向 SDE 具有相同的边缘分布 $q_t$。

所以：

$$
\boxed{
p_\theta^{\mathrm{ODE}}=q_0
}
$$

最终：

$$
\boxed{
p_\theta^{\mathrm{SDE}}
=
p_\theta^{\mathrm{ODE}}
=
q_0
}
$$

这就证明了 Theorem 2 的全部结论。

**注意：这里要求网络输出是某个完整扩散过程的真实边缘 Score，而不是仅仅要求它在每个时间点看起来像一个梯度场。**

因为任意一组时间相关的 Score 场，不一定满足同一个前向 SDE 的 Fokker–Planck 演化关系。

这也是论文明确强调的限制。[arXiv](https://arxiv.org/html/2101.09258?utm_source=chatgpt.com)

---

# 五、Theorem 3：逐样本 ELBO 的完整推导

这是论文中与 DDPM 联系最紧密的一部分。

Theorem 1 证明的是整体分布的 KL 上界：

$$
D_{\mathrm{KL}}(p_0\|p_\theta^{\mathrm{SDE}})
$$

但我们还希望对每个具体样本 $x_0$ 得到：

$$
-\log p_\theta^{\mathrm{SDE}}(x_0)
$$

的上界。

论文 Theorem 3 给出：

$$
\boxed{
-\log p_\theta^{\mathrm{SDE}}(x_0)
\leq
\mathcal L_\theta^{\mathrm{SM}}(x_0)
=
\mathcal L_\theta^{\mathrm{DSM}}(x_0)
}
\tag{11}
$$

这里有一个重要区别：

**Theorem 3 中的 SM 和 DSM 形式严格相等，因为 DSM 表达式额外包含了修正常数项。**

这不同于之前只用于训练的：

$$
\mathcal J_{\mathrm{SM}}
$$

和：

$$
\mathcal J_{\mathrm{DSM}}
$$

它们通常只相差一个常数。

## 15. 先写出最终公式

令：

$$
X_t\sim p_{0t}(\cdot\mid x_0)
$$

论文给出的 Score Matching 形式为：

$$
\boxed{
\begin{aligned}
\mathcal L_\theta^{\mathrm{SM}}(x_0)
={}&
-\mathbb E[\log\pi(X_T)\mid X_0=x_0]\\
&+
\frac12\int_0^T
\mathbb E\Big[
2g(t)^2\nabla\cdot s_\theta(X_t,t)\\
&\qquad
+g(t)^2\|s_\theta(X_t,t)\|^2
-2\nabla\cdot f(X_t,t)
\mid X_0=x_0
\Big]dt
\end{aligned}
}
$$

对应的 DSM 形式为：

$$
\boxed{
\begin{aligned}
\mathcal L_\theta^{\mathrm{DSM}}(x_0)
={}&
-\mathbb E[\log\pi(X_T)\mid x_0]\\
&+
\frac12\int_0^T
\mathbb E\Big[
g(t)^2\|s_\theta(X_t,t)-u_t(X_t,x_0)\|^2\\
&\qquad
-g(t)^2\|u_t(X_t,x_0)\|^2
-2\nabla\cdot f(X_t,t)
\mid x_0
\Big]dt
\end{aligned}
}
$$

其中：

$$
\boxed{
u_t(x_t,x_0)
=
\nabla_{x_t}\log p_{0t}(x_t\mid x_0)
}
$$

这两个表达式都来自论文第 4.2 节。[arXiv](https://arxiv.org/html/2101.09258?utm_source=chatgpt.com)

下面推导它们为什么相等，以及为什么它们能够上界负对数似然。

## 16. 第一步：从交叉熵出发

任意真实数据分布 $p_0$ 都满足：

$$
\boxed{
-\mathbb E_{p_0}\log p_\theta^{\mathrm{SDE}}
=
D_{\mathrm{KL}}(p_0\|p_\theta^{\mathrm{SDE}})
+
\mathcal H(p_0)
}
$$

其中：

$$
\mathcal H(p_0)
=
-\mathbb E_{p_0}\log p_0
$$

是微分熵。

Theorem 1 已经给出了 KL 上界：

$$
D_{\mathrm{KL}}(p_0\|p_\theta^{\mathrm{SDE}})
\leq
D_{\mathrm{KL}}(p_T\|\pi)
+
\frac12\int_0^Tg^2
\mathbb E_{p_t}\|s^*-s_\theta\|^2dt
$$

所以：

$$
\begin{aligned}
-\mathbb E_{p_0}\log p_\theta^{\mathrm{SDE}}
\leq{}&
D_{\mathrm{KL}}(p_T\|\pi)
+
\mathcal H(p_0)\\
&+
\frac12\int_0^Tg^2
\mathbb E_{p_t}\|s^*-s_\theta\|^2dt
\end{aligned}
$$

问题是：

$$
\mathcal H(p_0)
$$

未知。

因此需要推导熵在 SDE 下的演化规律。

---

## 17. 第二步：推导熵的时间导数

定义：

$$
\mathcal H(p_t)
=
-\int p_t(x)\log p_t(x)dx
$$

对时间求导：

$$
\begin{aligned}
\frac{d}{dt}\mathcal H(p_t)
&=
-\int\partial_tp_t(x)[\log p_t(x)+1]dx
\end{aligned}
$$

因为：

$$
\int\partial_tp_tdx=0
$$

所以：

$$
\boxed{
\frac{d}{dt}\mathcal H(p_t)
=
-\int\partial_tp_t\log p_tdx
}
$$

代入 Fokker–Planck：

$$
\partial_tp_t
=
-\nabla\cdot(fp_t)+\frac12g^2\Delta p_t
$$

得到：

$$
\begin{aligned}
\frac{d}{dt}\mathcal H(p_t)
={}&
\int\nabla\cdot(fp_t)\log p_tdx\\
&-
\frac12g^2\int\Delta p_t\log p_tdx
\end{aligned}
$$

### 17.1 处理漂移项

对第一项分部积分：

$$
\int\nabla\cdot(fp_t)\log p_tdx
=
-\int fp_t\cdot\nabla\log p_tdx
$$

即：

$$
-\mathbb E_{p_t}
[f^\top\nabla\log p_t]
$$

进一步：

$$
\begin{aligned}
\mathbb E_{p_t}[f^\top\nabla\log p_t]
&=
\int f^\top\nabla p_tdx\\
&=
-\int p_t\nabla\cdot fdx
\end{aligned}
$$

因此：

$$
\boxed{
-\mathbb E_{p_t}[f^\top\nabla\log p_t]
=
\mathbb E_{p_t}[\nabla\cdot f]
}
$$

### 17.2 处理扩散项

考虑：

$$
-\frac12g^2\int\Delta p_t\log p_tdx
$$

分部积分：

$$
\int\Delta p_t\log p_tdx
=
-\int\nabla p_t^\top\nabla\log p_tdx
$$

因为：

$$
\nabla p_t=p_t\nabla\log p_t
$$

所以：

$$
\int\Delta p_t\log p_tdx
=
-\int p_t\|\nabla\log p_t\|^2dx
$$

因此：

$$
\boxed{
-\frac12g^2\int\Delta p_t\log p_tdx
=
\frac12g^2
\mathbb E_{p_t}\|\nabla\log p_t\|^2
}
$$

最终得到：

$$
\boxed{
\frac{d}{dt}\mathcal H(p_t)
=
\mathbb E_{p_t}[\nabla\cdot f]
+
\frac12g(t)^2
\mathbb E_{p_t}\|\nabla\log p_t\|^2
}
$$

这就是论文 Theorem 4 中的熵演化公式。

### 17.3 为什么它与 de Bruijn 恒等式有关？

如果前向 SDE 是纯加性噪声：

$$
dX_t=dW_t
$$

则：

$$
f=0,\qquad g=1
$$

因此：

$$
\boxed{
\frac{d}{dt}\mathcal H(p_t)
=
\frac12
\mathbb E_{p_t}\|\nabla\log p_t\|^2
}
$$

这正是经典 de Bruijn 恒等式。

它揭示：

**高斯扩散使分布熵增加，而熵增加的速度由 Fisher 信息决定。**

---

## 18. 第三步：将初始熵写成终端熵

对熵演化公式积分：

$$
\mathcal H(p_T)-\mathcal H(p_0)
=
\int_0^T
\left[
\mathbb E_{p_t}\nabla\cdot f
+
\frac12g^2
\mathbb E_{p_t}\|s^*\|^2
\right]dt
$$

整理：

$$
\boxed{
\begin{aligned}
\mathcal H(p_0)
={}&
\mathcal H(p_T)\\
&-
\int_0^T
\mathbb E_{p_t}[\nabla\cdot f]dt\\
&-
\frac12\int_0^T
g(t)^2\mathbb E_{p_t}\|s^*\|^2dt
\end{aligned}
}
$$

其中：

$$
s^*=\nabla\log p_t
$$

这一步非常关键，因为它将未知的初始熵转换成了终端熵和积分项。

---

## 19. 第四步：与 KL 上界合并

回到：

$$
\begin{aligned}
-\mathbb E_{p_0}\log p_\theta^{\mathrm{SDE}}
\leq{}&
D_{\mathrm{KL}}(p_T\|\pi)
+\mathcal H(p_0)\\
&+
\frac12\int_0^Tg^2
\mathbb E_{p_t}\|s^*-s_\theta\|^2dt
\end{aligned}
$$

代入初始熵公式：

$$
\begin{aligned}
-\mathbb E_{p_0}\log p_\theta^{\mathrm{SDE}}
\leq{}&
D_{\mathrm{KL}}(p_T\|\pi)
+\mathcal H(p_T)\\
&+
\frac12\int_0^Tg^2
\mathbb E_{p_t}
\left[
\|s^*-s_\theta\|^2-\|s^*\|^2
\right]dt\\
&-
\int_0^T
\mathbb E_{p_t}[\nabla\cdot f]dt
\end{aligned}
$$

先处理前两项：

$$
\begin{aligned}
D_{\mathrm{KL}}(p_T\|\pi)+\mathcal H(p_T)
&=
\mathbb E_{p_T}
\left[
\log\frac{p_T}{\pi}
\right]
-
\mathbb E_{p_T}\log p_T\\
&=
-\mathbb E_{p_T}\log\pi
\end{aligned}
$$

因此：

$$
\boxed{
\begin{aligned}
-\mathbb E_{p_0}\log p_\theta^{\mathrm{SDE}}
\leq{}&
-\mathbb E_{p_T}\log\pi\\
&+
\frac12\int_0^Tg^2
\mathbb E_{p_t}
\left[
\|s^*-s_\theta\|^2-\|s^*\|^2
\right]dt\\
&-
\int_0^T\mathbb E_{p_t}[\nabla\cdot f]dt
\end{aligned}
}
\tag{E}
$$

这个公式已经非常接近 Theorem 3。

---

## 20. 第五步：将 Score Matching 改写为散度形式

考虑：

$$
\|s^*-s_\theta\|^2-\|s^*\|^2
$$

展开：

$$
\begin{aligned}
\|s^*-s_\theta\|^2-\|s^*\|^2
&=
\|s_\theta\|^2-2s_\theta^\top s^*
\end{aligned}
$$

所以：

$$
\begin{aligned}
\mathbb E_{p_t}
\left[
\|s^*-s_\theta\|^2-\|s^*\|^2
\right]
=
\mathbb E_{p_t}\|s_\theta\|^2
-
2\mathbb E_{p_t}[s_\theta^\top s^*]
\end{aligned}
$$

现在：

$$
s^*=\nabla\log p_t
$$

因此：

$$
\begin{aligned}
\mathbb E_{p_t}[s_\theta^\top s^*]
&=
\int p_ts_\theta^\top\nabla\log p_tdx\\
&=
\int s_\theta^\top\nabla p_tdx
\end{aligned}
$$

分部积分：

$$
\int s_\theta^\top\nabla p_tdx
=
-\int p_t\nabla\cdot s_\theta dx
$$

因此：

$$
\boxed{
\mathbb E_{p_t}[s_\theta^\top s^*]
=
-\mathbb E_{p_t}[\nabla\cdot s_\theta]
}
$$

代回：

$$
\boxed{
\begin{aligned}
\mathbb E_{p_t}
\left[
\|s^*-s_\theta\|^2-\|s^*\|^2
\right]
=
\mathbb E_{p_t}
\left[
\|s_\theta\|^2+2\nabla\cdot s_\theta
\right]
\end{aligned}
}
$$

最终得到：

$$
\boxed{
\begin{aligned}
-\mathbb E_{p_0}\log p_\theta^{\mathrm{SDE}}
\leq{}&
-\mathbb E_{p_T}\log\pi\\
&+
\frac12\int_0^T
\mathbb E_{p_t}
\left[
g(t)^2\|s_\theta\|^2
+2g(t)^2\nabla\cdot s_\theta
-2\nabla\cdot f
\right]dt
\end{aligned}
}
$$

这就是 Theorem 3 的 SM 形式在数据分布上的期望版本。

---

## 21. 第六步：为什么可以得到逐样本上界？

前面推导的是：

$$
-\mathbb E_{p_0}\log p_\theta^{\mathrm{SDE}}
\leq
\mathbb E_{p_0}\mathcal L_\theta^{\mathrm{SM}}(X_0)
$$

但是：

$$
\mathbb E[A]\leq\mathbb E[B]
$$

一般不能推出：

$$
A(x)\leq B(x)
$$

对每个 $x$ 都成立。

因此不能简单地“去掉期望”。

论文这里使用了一个更巧妙的思路：

**前面的分布级推导对于任意满足条件的初始分布 $p_0$ 都成立。**

如果对某个点：

$$
x_0
$$

有：

$$
-\log p_\theta^{\mathrm{SDE}}(x_0)
>
\mathcal L_\theta^{\mathrm{SM}}(x_0)
$$

并且该不等式在某个邻域内持续成立，那么可以构造一个集中在该邻域内的平滑初始分布。

对这个分布应用刚刚证明的期望上界，就会产生矛盾。

因此，在相应连续性和极限交换条件下，可以推出逐样本上界：

$$
\boxed{
-\log p_\theta^{\mathrm{SDE}}(x_0)
\leq
\mathcal L_\theta^{\mathrm{SM}}(x_0)
}
$$

这一步依赖上界对任意初始分布成立，而不是普通的期望不等式。

---

## 22. 第七步：将 SM 形式转换成 DSM 形式

固定一个干净样本：

$$
x_0
$$

考虑条件分布：

$$
p_{0t}(x_t\mid x_0)
$$

定义条件 Score：

$$
u_t(x_t,x_0)
=
\nabla_{x_t}\log p_{0t}(x_t\mid x_0)
$$

我们希望将：

$$
\mathbb E_{p_{0t}}
\left[
\|s_\theta\|^2+2\nabla\cdot s_\theta
\right]
$$

转换成 DSM 形式。

根据分部积分：

$$
\begin{aligned}
\mathbb E_{p_{0t}}[s_\theta^\top u_t]
&=
\int s_\theta^\top\nabla p_{0t}dx_t\\
&=
-\int p_{0t}\nabla\cdot s_\theta dx_t
\end{aligned}
$$

因此：

$$
\boxed{
\mathbb E_{p_{0t}}[s_\theta^\top u_t]
=
-\mathbb E_{p_{0t}}[\nabla\cdot s_\theta]
}
$$

另一方面：

$$
\|s_\theta-u_t\|^2
=
\|s_\theta\|^2+\|u_t\|^2-2s_\theta^\top u_t
$$

所以：

$$
\|s_\theta-u_t\|^2-\|u_t\|^2
=
\|s_\theta\|^2-2s_\theta^\top u_t
$$

取期望：

$$
\boxed{
\begin{aligned}
\mathbb E_{p_{0t}}
\left[
\|s_\theta-u_t\|^2-\|u_t\|^2
\right]
=
\mathbb E_{p_{0t}}
\left[
\|s_\theta\|^2+2\nabla\cdot s_\theta
\right]
\end{aligned}
}
$$

因此，Theorem 3 的两个表达式严格相等：

$$
\boxed{
\mathcal L_\theta^{\mathrm{SM}}(x_0)
=
\mathcal L_\theta^{\mathrm{DSM}}(x_0)
}
$$

最终：

$$
\boxed{
-\log p_\theta^{\mathrm{SDE}}(x_0)
\leq
\mathcal L_\theta^{\mathrm{DSM}}(x_0)
}
$$

这就是连续时间扩散模型的逐样本变分上界。

推导对应论文附录 A 的 Theorem 4、Theorem 5，以及第 4.2 节的 Theorem 3。[arXiv](https://arxiv.org/html/2101.09258?utm_source=chatgpt.com)

---

# 六、为什么 Theorem 3 可以理解为连续时间 DDPM ELBO？

我们将 DDPM 和 ScoreFlow 的损失放在一起比较。

## 23. DDPM 的离散 ELBO

DDPM 的负 ELBO 为：

$$
\boxed{
\begin{aligned}
\mathcal L_{\mathrm{DDPM}}
={}&
D_{\mathrm{KL}}(q(x_T\mid x_0)\|p(x_T))\\
&+
\sum_{k=2}^T
\mathbb E_{q(x_k\mid x_0)}
D_{\mathrm{KL}}
\left(
q(x_{k-1}\mid x_k,x_0)
\|
p_\theta(x_{k-1}\mid x_k)
\right)\\
&-
\mathbb E_{q(x_1\mid x_0)}
\log p_\theta(x_0\mid x_1)
\end{aligned}
}
$$

核心中间项来自高斯 KL：

$$
D_{\mathrm{KL}}(q\|p_\theta)
=
\frac1{2\sigma_k^2}
\mathbb E\|\tilde\mu_k-\mu_\theta\|^2+C
$$

代入噪声参数化：

$$
\boxed{
\mathcal L_k
=
\frac{\beta_k^2}
{2\sigma_k^2\alpha_k(1-\bar\alpha_k)}
\mathbb E\|\epsilon-\epsilon_\theta\|^2+C
}
$$

## 24. ScoreFlow 的连续时间 ELBO

考虑 VP-SDE：

$$
dX_t=
-\frac12\beta(t)X_tdt+\sqrt{\beta(t)}dW_t
$$

所以：

$$
g(t)^2=\beta(t)
$$

定义：

$$
B(t)=\int_0^t\beta(s)ds
$$

条件转移分布：

$$
X_t=
e^{-B(t)/2}X_0+
\sqrt{1-e^{-B(t)}}\epsilon
$$

记：

$$
\sigma_t^2=1-e^{-B(t)}
$$

条件 Score：

$$
u_t=-\frac{\epsilon}{\sigma_t}
$$

如果网络使用噪声参数化：

$$
s_\theta(x_t,t)
=
-\frac{\epsilon_\theta(x_t,t)}{\sigma_t}
$$

那么：

$$
s_\theta-u_t
=
-\frac{\epsilon_\theta-\epsilon}{\sigma_t}
$$

因此：

$$
\|s_\theta-u_t\|^2
=
\frac{\|\epsilon_\theta-\epsilon\|^2}{\sigma_t^2}
$$

代入 Likelihood Weighting：

$$
\frac12g(t)^2\|s_\theta-u_t\|^2
$$

得到：

$$
\boxed{
\frac{\beta(t)}
{2[1-e^{-B(t)}]}
\|\epsilon_\theta-\epsilon\|^2
}
$$

所以 ScoreFlow 的可训练部分为：

$$
\boxed{
\mathcal L_{\mathrm{train}}
=
\frac12\int_0^T
\frac{\beta(t)}
{1-e^{-B(t)}}
\mathbb E\|\epsilon_\theta-\epsilon\|^2dt
}
$$

这就是连续时间下的似然加权噪声预测目标。

## 25. 与 DDPM 的离散权重对比

DDPM：

$$
w_k=
\frac{\beta_k^2}
{2\sigma_k^2\alpha_k(1-\bar\alpha_k)}
$$

假设采用：

$$
\sigma_k^2=\beta_k
$$

那么：

$$
w_k=
\frac{\beta_k}
{2\alpha_k(1-\bar\alpha_k)}
$$

令：

$$
\beta_k=\beta(t_k)\Delta t
$$

当：

$$
\Delta t\rightarrow0
$$

时：

$$
\alpha_k=1-\beta_k\rightarrow1
$$

并且：

$$
\bar\alpha_k\rightarrow e^{-B(t_k)}
$$

因此：

$$
w_k
\approx
\frac{
\beta(t_k)\Delta t
}{
2[1-e^{-B(t_k)}]
}
$$

于是：

$$
\sum_kw_k\mathbb E\|\epsilon-\epsilon_\theta\|^2
$$

趋向：

$$
\boxed{
\frac12\int_0^T
\frac{\beta(t)}
{1-e^{-B(t)}}
\mathbb E\|\epsilon-\epsilon_\theta\|^2dt
}
$$

这与 ScoreFlow 的连续时间目标一致。

**因此，Likelihood Weighting 可以理解为 DDPM 离散 KL 权重在连续时间下的对应形式。**

但要注意，这里比较的是内部去噪 KL 项。完整 ELBO 还包括终端先验项和重建项，不能仅凭这一项就认定整个离散 ELBO 与连续公式完全相同。

---

# 七、论文第 5.1 节：Importance Sampling 的完整推导

## 26. 为什么 Likelihood Weighting 会导致高方差？

ScoreFlow 的目标为：

$$
\boxed{
J(\theta)
=
\frac12\int_0^Tg(t)^2
\mathbb E[\ell_\theta(t)]dt
}
$$

其中：

$$
\ell_\theta(t)
=
\|s_\theta(X_t,t)-u_t(X_t,X_0)\|^2
$$

通常通过均匀采样时间：

$$
t\sim U[0,T]
$$

来估计积分。

因为均匀分布的密度是：

$$
\rho_{\mathrm{uni}}(t)=\frac1T
$$

所以：

$$
\begin{aligned}
J(\theta)
&=
\frac12
\mathbb E_{t\sim U[0,T]}
\left[
Tg(t)^2\mathbb E[\ell_\theta(t)]
\right]
\end{aligned}
$$

对应单样本估计：

$$
\boxed{
\hat J_{\mathrm{uni}}
=
\frac T2g(t)^2\ell_\theta(t)
}
$$

这个估计是无偏的，但可能具有较大方差。

### 26.1 为什么小噪声时间容易出现问题？

以 VP-SDE 为例：

$$
g(t)^2=\beta(t)
$$

条件噪声方差：

$$
\sigma_t^2=1-e^{-B(t)}
$$

条件 Score：

$$
u_t=-\frac{\epsilon}{\sigma_t}
$$

如果网络刚开始训练，预测误差较大，那么：

$$
\ell_\theta(t)
$$

可能包含：

$$
\frac1{\sigma_t^2}
$$

量级的项。

当：

$$
t\rightarrow0
$$

时：

$$
\sigma_t^2\rightarrow0
$$

于是损失可能很大。

不同时间的损失量级差异会导致 Monte Carlo 估计方差增大。

这也是为什么论文实际训练时使用：

$$
t\in[\varepsilon,T]
$$

避免严格的零噪声端点。

## 27. 重要性采样的一般公式

考虑积分：

$$
I=\int_0^TF(t)dt
$$

任意选择概率密度：

$$
\rho(t)>0
$$

满足：

$$
\int_0^T\rho(t)dt=1
$$

那么：

$$
\begin{aligned}
I
&=
\int_0^T
\rho(t)\frac{F(t)}{\rho(t)}dt\\
&=
\mathbb E_{t\sim\rho}
\left[
\frac{F(t)}{\rho(t)}
\right]
\end{aligned}
$$

因此：

$$
\boxed{
I=
\mathbb E_{t\sim\rho}
\left[
\frac{F(t)}{\rho(t)}
\right]
}
$$

这就是 Importance Sampling。

它不会改变积分本身，只是改变采样时间的分布和对应权重。

## 28. ScoreFlow 如何选择时间采样密度？

论文原来的时间权重记为：

$$
\lambda_{\mathrm{old}}(t)
$$

对于 VP-SDE：

$$
\lambda_{\mathrm{old}}(t)=\sigma_t^2
$$

而 Likelihood Weighting 为：

$$
\lambda_{\mathrm{ML}}(t)=g(t)^2
$$

作者希望：

**通过改变时间采样概率，让每次训练中的有效损失权重重新接近旧权重，但整体优化目标仍然是 Likelihood Weighting。**

因此选择：

$$
\boxed{
\rho(t)
=
\frac{g(t)^2}
{Z\lambda_{\mathrm{old}}(t)}
}
$$

其中：

$$
\boxed{
Z=
\int_\varepsilon^T
\frac{g(t)^2}{\lambda_{\mathrm{old}}(t)}dt
}
$$

是归一化常数。

于是：

$$
\begin{aligned}
J(\theta)
&=
\frac12\int_\varepsilon^T
g(t)^2\mathbb E[\ell_\theta(t)]dt\\
&=
\frac12\int_\varepsilon^T
\rho(t)
\frac{g(t)^2}{\rho(t)}
\mathbb E[\ell_\theta(t)]dt
\end{aligned}
$$

因为：

$$
\rho(t)=
\frac{g(t)^2}{Z\lambda_{\mathrm{old}}(t)}
$$

所以：

$$
\frac{g(t)^2}{\rho(t)}
=
Z\lambda_{\mathrm{old}}(t)
$$

最终：

$$
\boxed{
J(\theta)
=
\frac Z2
\mathbb E_{t\sim\rho}
\left[
\lambda_{\mathrm{old}}(t)
\ell_\theta(t)
\right]
}
$$

这样就得到了一个新的无偏估计器：

$$
\boxed{
\hat J_{\mathrm{IS}}
=
\frac Z2
\lambda_{\mathrm{old}}(t)
\ell_\theta(t),
\qquad t\sim\rho
}
$$

这就是论文第 5.1 节的核心思想。

注意：这里使用的是归一化概率密度 $\rho$，因此按照标准积分定义，前面的系数是 $Z$，不需要额外乘 $T$。论文公式 (12) 的展示中存在一个额外的 $T$ 因子；理解和实现时应首先明确采样密度的归一化约定。[arXiv](https://arxiv.org/html/2101.09258?utm_source=chatgpt.com)

---

## 29. VP-SDE 的重要性采样密度

VP-SDE：

$$
g(t)^2=\beta(t)
$$

旧权重：

$$
\lambda_{\mathrm{old}}(t)=1-e^{-B(t)}
$$

因此：

$$
\boxed{
\rho(t)
\propto
\frac{\beta(t)}
{1-e^{-B(t)}}
}
$$

这个密度在较小时间 $t$ 处通常更大。

为什么？

因为：

$$
B(t)\approx\beta(0)t
$$

所以：

$$
1-e^{-B(t)}
\approx B(t)\approx\beta(0)t
$$

因此：

$$
\frac{\beta(t)}{1-e^{-B(t)}}
\approx
\frac1t
$$

这意味着重要性采样会更频繁地抽取小噪声时间。

但同时，每次训练时乘上的有效权重是：

$$
\sigma_t^2
$$

从而降低单次损失的尺度波动。

### 29.1 这个分布为什么必须截断？

因为：

$$
\rho(t)\propto\frac1t
$$

在：

$$
t\rightarrow0
$$

附近不可积。

因此必须使用：

$$
\varepsilon>0
$$

并在：

$$
[\varepsilon,T]
$$

上归一化。

这不仅是数值稳定问题，也是使该重要性采样密度可归一化的必要条件。

---

## 30. 如何直接采样 VP-SDE 的重要性时间？

对于 VP-SDE：

$$
\rho(t)
\propto
\frac{\beta(t)}
{1-e^{-B(t)}}
$$

注意：

$$
B'(t)=\beta(t)
$$

考虑函数：

$$
F(t)=\log(e^{B(t)}-1)
$$

求导：

$$
\begin{aligned}
F'(t)
&=
\frac{e^{B(t)}B'(t)}
{e^{B(t)}-1}\\
&=
\frac{\beta(t)}
{1-e^{-B(t)}}
\end{aligned}
$$

因此：

$$
\boxed{
\rho(t)\propto F'(t)
}
$$

于是累积分布函数：

$$
P(t'\leq t)
=
\frac{F(t)-F(\varepsilon)}
{F(T)-F(\varepsilon)}
$$

令：

$$
u\sim U(0,1)
$$

那么：

$$
F(t)=F(\varepsilon)+u[F(T)-F(\varepsilon)]
$$

即：

$$
\log(e^{B(t)}-1)
=
F(\varepsilon)+u[F(T)-F(\varepsilon)]
$$

两边取指数：

$$
e^{B(t)}-1
=
\exp\left(
F(\varepsilon)+u[F(T)-F(\varepsilon)]
\right)
$$

所以：

$$
\boxed{
B(t)
=
\log\left[
1+\exp\left(
F(\varepsilon)+u[F(T)-F(\varepsilon)]
\right)
\right]
}
$$

如果：

$$
\beta(t)=\beta_{\min}
+t(\beta_{\max}-\beta_{\min})
$$

那么：

$$
B(t)=\beta_{\min}t+
\frac12(\beta_{\max}-\beta_{\min})t^2
$$

记：

$$
\Delta\beta=\beta_{\max}-\beta_{\min}
$$

解二次方程：

$$
\frac12\Delta\beta t^2+\beta_{\min}t-B=0
$$

得到：

$$
\boxed{
t=
\frac{
-\beta_{\min}
+
\sqrt{\beta_{\min}^2+2\Delta\beta B}
}{
\Delta\beta
}
}
$$

因此，VP-SDE 的重要性时间可以直接通过逆 CDF 采样，不需要拒绝采样。

这对后续实现代码非常有用。

---

# 八、论文第 5.2 节：Variational Dequantization

这是论文中容易被忽略、但对最终似然指标很重要的部分。

## 31. 为什么图像生成模型需要去量化？

原始图像通常是离散整数：

$$
x\in\{0,1,\ldots,255\}^D
$$

但 SDE 模型学习的是连续概率密度：

$$
p_\theta(y)
$$

其中：

$$
y\in\mathbb R^D
$$

连续概率密度与离散概率质量不是同一个对象。

对于离散图像 $x$，正确的概率质量应该是对应量化区间的积分：

$$
\boxed{
P_\theta(x)
=
\int_{[0,1)^D}
p_\theta(x+u)\,du
}
$$

这里为了简化推导，先不考虑把像素归一化到 $[0,1]$ 后产生的尺度变换。

直接计算这个高维积分通常非常困难。

因此可以引入去量化噪声：

$$
u\in[0,1)^D
$$

将离散图像转化为连续变量：

$$
y=x+u
$$

---

## 32. Uniform Dequantization

最简单的方法是：

$$
u\sim U([0,1)^D)
$$

那么：

$$
y=x+u
$$

根据 Jensen 不等式：

$$
\begin{aligned}
\log P_\theta(x)
&=
\log\int_{[0,1)^D}p_\theta(x+u)du\\
&=
\log\mathbb E_{u\sim U}
[p_\theta(x+u)]\\
&\geq
\mathbb E_{u\sim U}
[\log p_\theta(x+u)]
\end{aligned}
$$

所以：

$$
\boxed{
\log P_\theta(x)
\geq
\mathbb E_{u\sim U}
[\log p_\theta(x+u)]
}
$$

这说明：训练连续模型的去量化对数似然，可以最大化离散图像对数似然的一个下界。

但这个下界可能不够紧。

---

## 33. Variational Dequantization

论文进一步采用可学习的去量化分布：

$$
q_\phi(u\mid x)
$$

其中：

$$
u\in[0,1)^D
$$

现在：

$$
P_\theta(x)
=
\int q_\phi(u\mid x)
\frac{p_\theta(x+u)}{q_\phi(u\mid x)}du
$$

因此：

$$
\log P_\theta(x)
=
\log\mathbb E_{q_\phi}
\left[
\frac{p_\theta(x+u)}{q_\phi(u\mid x)}
\right]
$$

根据 Jensen：

$$
\boxed{
\log P_\theta(x)
\geq
\mathbb E_{q_\phi}
\left[
\log p_\theta(x+u)
-
\log q_\phi(u\mid x)
\right]
}
$$

这就是论文公式 (13)。

定义：

$$
\mathcal L_{\mathrm{deq}}
=
\mathbb E_{q_\phi}
\left[
\log p_\theta(x+u)
-
\log q_\phi(u\mid x)
\right]
$$

我们还可以进一步理解这个下界的松弛程度。

定义真实的去量化后验：

$$
\boxed{
p_\theta(u\mid x)
=
\frac{p_\theta(x+u)}
{P_\theta(x)}
}
$$

其中 $u\in[0,1)^D$。

计算：

$$
D_{\mathrm{KL}}(q_\phi(u\mid x)\|p_\theta(u\mid x))
$$

展开：

$$
\begin{aligned}
D_{\mathrm{KL}}
&=
\mathbb E_{q_\phi}
\left[
\log q_\phi(u\mid x)
-
\log p_\theta(u\mid x)
\right]\\
&=
\mathbb E_{q_\phi}
\left[
\log q_\phi(u\mid x)
-
\log p_\theta(x+u)
+
\log P_\theta(x)
\right]
\end{aligned}
$$

整理：

$$
\boxed{
\log P_\theta(x)
-
\mathcal L_{\mathrm{deq}}
=
D_{\mathrm{KL}}
\left(
q_\phi(u\mid x)
\|
p_\theta(u\mid x)
\right)
}
$$

因此：

**Variational Dequantization 的作用，就是学习更接近真实去量化后验的噪声分布，从而减小离散似然下界的松弛程度。**

### 33.1 与 ScoreFlow 的逐样本上界结合

Theorem 3 给出：

$$
-\log p_\theta^{\mathrm{SDE}}(y)
\leq
\mathcal L_\theta^{\mathrm{DSM}}(y)
$$

因此：

$$
\log p_\theta^{\mathrm{SDE}}(y)
\geq
-\mathcal L_\theta^{\mathrm{DSM}}(y)
$$

将它代入去量化下界：

$$
\begin{aligned}
\log P_\theta(x)
&\geq
\mathbb E_{q_\phi}
\left[
\log p_\theta^{\mathrm{SDE}}(x+u)
-
\log q_\phi(u\mid x)
\right]\\
&\geq
\mathbb E_{q_\phi}
\left[
-\mathcal L_\theta^{\mathrm{DSM}}(x+u)
-
\log q_\phi(u\mid x)
\right]
\end{aligned}
$$

所以可以最大化：

$$
\boxed{
\mathbb E_{q_\phi}
\left[
-\mathcal L_\theta^{\mathrm{DSM}}(x+u)
-
\log q_\phi(u\mid x)
\right]
}
$$

这就是论文将连续时间 DSM 与 Variational Dequantization 结合起来的数学依据。[arXiv](https://arxiv.org/html/2101.09258?utm_source=chatgpt.com)

---

# 九、论文第 4.3 节与附录 B：数值稳定性

## 34. 为什么实际训练不从 $t=0$ 开始？

对于 VP-SDE：

$$
X_t=e^{-B(t)/2}X_0+
\sqrt{1-e^{-B(t)}}\epsilon
$$

当：

$$
t\rightarrow0
$$

时：

$$
B(t)\rightarrow0
$$

因此：

$$
\sigma_t^2=1-e^{-B(t)}\rightarrow0
$$

条件 Score：

$$
u_t=-\frac{\epsilon}{\sigma_t}
$$

可能出现很大的数值。

同时 Likelihood Weighting 下的噪声预测损失权重：

$$
\frac{\beta(t)}{\sigma_t^2}
$$

也可能趋于无穷。

所以论文实际使用：

$$
\boxed{
t\in[\varepsilon,T]
}
$$

其中：

$$
\varepsilon>0
$$

这可以避免在零噪声端点直接计算不稳定的 Score。

但这也意味着：

**实际计算的连续时间似然上界对应截断后的扩散过程，不能不加修正就直接当成原始 $t=0$ 数据模型的似然上界。**

论文附录 B 讨论了通过 Jensen 不等式修正这一问题的方法。

## 35. 截断时间带来的问题

假设我们只模拟：

$$
T\rightarrow\varepsilon
$$

得到模型在时间 $\varepsilon$ 的密度：

$$
p_{\theta,\varepsilon}(x_\varepsilon)
$$

但我们希望评估：

$$
p_{\theta,0}(x_0)
$$

如果没有定义：

$$
\varepsilon\rightarrow0
$$

的最后一段生成过程，就不能直接将两者视为相同。

论文通过额外的概率建模和 Jensen 下界处理这段差异，使结果能够用于可比较的似然评估。

这是一个数值实现问题，但它对 bits/dim 的正确计算很重要。

---

# 十、论文实验部分：哪些结论真正得到验证？

作者使用：

- CIFAR-10。
- ImageNet $32\times32$。
- VP-SDE。
- sub-VP-SDE。
- DDPM++。
- DDPM++ deep。

对比了不同训练配置：

1. 原始 Score Matching 权重。
2. Likelihood Weighting。
3. Likelihood Weighting + Importance Sampling。
4. 再结合 Variational Dequantization。

主要结论如下。

### 36. Likelihood Weighting 能否改善似然？

论文发现，在 CIFAR-10 上，Likelihood Weighting 通常能改善似然。

但在 ImageNet $32\times32$ 上，单独使用 Likelihood Weighting 并不能保证所有配置都得到改善。

加入 Importance Sampling 后，实验表现更加稳定。

这说明：

**理论上更符合最大似然的目标，不代表使用高方差优化器时一定能够获得更好的训练结果。**

### 37. Importance Sampling 的效果

论文报告：

CIFAR-10 上，训练损失方差从：

$$
98.48
$$

下降到：

$$
0.068
$$

ImageNet 上，从：

$$
0.51
$$

下降到：

$$
0.043
$$

这说明重要性采样能够显著改善训练目标的数值稳定性。

### 38. 最终似然指标

论文最终报告：

$$
\boxed{
\mathrm{NLL}_{\mathrm{CIFAR10}}
=
2.83\ \mathrm{bits/dim}
}
$$

$$
\boxed{
\mathrm{NLL}_{\mathrm{ImageNet32}}
=
3.76\ \mathrm{bits/dim}
}
$$

这两个结果是在不使用数据增强的配置下获得的。

作者还观察到：

**提高似然有时会导致 FID 略微变差。**

这是生成模型中一个非常重要的现象。

似然衡量模型为真实数据分配的概率，而 FID 衡量生成样本与真实样本在特征空间中的统计差异。

两者不是同一个优化目标，因此并不保证同步改善。[arXiv](https://arxiv.org/html/2101.09258?utm_source=chatgpt.com)

---

# 十一、整篇论文最重要的理论关系

我们可以将这篇论文的三个定理串起来理解。

## 39. Theorem 1：KL 上界

$$
\boxed{
D_{\mathrm{KL}}(p_0\|p_\theta^{\mathrm{SDE}})
\leq
D_{\mathrm{KL}}(p_T\|\pi)
+
\frac12\int_0^Tg(t)^2
\mathbb E_{p_t}
\left[
\|\nabla\log p_t-s_\theta\|^2
\right]dt
}
$$

它告诉我们：

**为什么 Likelihood Weighting 是 $g(t)^2$。**

## 40. Theorem 2：KL 等式

当：

$$
s_\theta=\nabla\log q_t
$$

且：

$$
q_T=\pi
$$

时：

$$
\boxed{
p_\theta^{\mathrm{SDE}}
=
p_\theta^{\mathrm{ODE}}
=
q_0
}
$$

并且：

$$
\boxed{
D_{\mathrm{KL}}(p_0\|q_0)
=
D_{\mathrm{KL}}(p_T\|q_T)
+
\frac12\int_0^Tg(t)^2
I(p_t\|q_t)dt
}
$$

它告诉我们：

**为什么 Score Matching 可以与 KL 散度建立精确联系，以及什么情况下 SDE 与 ODE 的模型分布相同。**

## 41. Theorem 3：逐样本 ELBO

$$
\boxed{
-\log p_\theta^{\mathrm{SDE}}(x_0)
\leq
\mathcal L_\theta^{\mathrm{DSM}}(x_0)
}
$$

它告诉我们：

**为什么连续时间 DSM 不仅能用于训练 Score，还能构造类似 DDPM ELBO 的逐样本似然下界。**

---

# 十二、结合你之前研究的 Flow Matching，再深入理解一步

这篇论文还有一个对后续研究很重要的启发。

我们之前知道，给定前向 SDE：

$$
dX_t=fdt+gdW_t
$$

可以构造 Probability Flow ODE：

$$
\frac{dX_t}{dt}
=
f-\frac12g^2\nabla\log p_t
$$

而 ScoreFlow 进一步说明：

**如果我们希望优化反向 SDE 生成分布的似然，那么 Score 误差在不同时间上的权重，应当与扩散系数平方 $g(t)^2$ 联系起来。**

但这并不意味着：

$$
\boxed{
\text{Flow Matching 速度 MSE}
=
\text{ScoreFlow Likelihood Weighting}
}
$$

因为 Flow Matching 直接学习：

$$
v_\theta(x,t)
$$

而 ScoreFlow 学习：

$$
s_\theta(x,t)
$$

二者通过：

$$
v=f-\frac12g^2s
$$

联系。

假设 $f,g$ 固定，并且我们使用：

$$
v_\theta=f-\frac12g^2s_\theta
$$

那么：

$$
v_\theta-v^*
=
-\frac12g^2(s_\theta-s^*)
$$

因此：

$$
\|s_\theta-s^*\|^2
=
\frac4{g^4}\|v_\theta-v^*\|^2
$$

代入 Likelihood Weighting：

$$
\frac12g^2\|s_\theta-s^*\|^2
$$

得到：

$$
\boxed{
\frac12g^2\|s_\theta-s^*\|^2
=
\frac2{g^2}\|v_\theta-v^*\|^2
}
$$

所以，在这一特定的 SDE–ODE 对应关系下，ScoreFlow 的似然加权 Score 损失可以改写为：

$$
\boxed{
\mathcal J_{\mathrm{ML}}
=
\int_0^T
\frac2{g(t)^2}
\mathbb E_{p_t}
\left[
\|v_\theta-v^*\|^2
\right]dt
}
$$

这个公式值得特别关注。

它说明：

**如果把 ScoreFlow 的训练目标转换到 Probability Flow ODE 的速度空间，那么对应的速度误差权重反而与 $1/g(t)^2$ 成正比。**

但必须强调，这里只是对特定速度参数化下的训练目标进行代数变换，并不意味着任意 Flow Matching 模型使用该权重就能获得相同的最大似然保证。

---

## 最后总结

这篇论文最重要的贡献，可以概括为：

$$
\boxed{
\text{Score Matching}
\xrightarrow[\lambda(t)=g(t)^2]{\text{路径 KL}}
\text{近似最大似然训练}
}
$$

其中有三条核心数学关系：

$$
\boxed{
\text{Girsanov}
\Rightarrow
\text{路径 KL 与 Score MSE 的关系}
}
$$

$$
\boxed{
\text{Fokker–Planck}
\Rightarrow
\text{KL 的时间导数与相对 Fisher 信息}
}
$$

$$
\boxed{
\text{熵演化 + KL 上界}
\Rightarrow
\text{连续时间逐样本 ELBO}
}
$$

与之前三篇论文相比，ScoreFlow 最大的进步并不是提出新的生成动力学，而是**解释了为什么特定的 Score Matching 目标具有最大似然意义**，并给出了可以实际优化的训练方法。

如果按我们之前 NCSN、DDPM、Song SDE 的学习路线继续，下一步最合适的实验是：**在相同的二维高斯混合分布上，对比普通 DSM 权重与 Likelihood Weighting，并计算 Probability Flow ODE 的真实模型 NLL，观察理论目标与实际似然的关系。**